#
<font color="tomato"> <center><h1>ISAT 449 – Emerging Topics in Applied Data Science </h1></center></font>


<font color="Orange"><center><h2>PathFinder: RAG-Based Academic Major Recommendation System</h2></center></font>


---

#### Author: A. Ananth

---

### **Objectives:**
- Many students enter the College of Integrated Science & Engineering (CISE) with broad interests in technology, computing, and societal impact, yet struggle to choose a major that aligns with their goals.

- Traditional advising resources can be overwhelming, and program descriptions are scattered across different departmental sites.
Students often ask questions like:

    - “I like computers but not Computer Science — what should I pick?”

    - “Which major focuses on science, technology, and society?”

    - “What concentrations exist within ISAT or Geography?”

- The purpose of this project is to build a Retrieval-Augmented Generation (RAG)–powered advising assistant that can:

    - Retrieve accurate descriptions of JMU CISE majors and concentrations

    - Match them to a student's interests

    - Provide reliable recommendations

    - Explain its reasoning using only the retrieved program text

- The assistant uses:

    - Program descriptions scraped from JMU websites

    - A ChromaDB vector store for semantic search

    - A lightweight LLM (Gemma 3 1B IT)

    - A custom prompt that prevents hallucinations and enforces valid major selection

Ultimately, this project demonstrates how RAG pipelines can be applied to real advising problems, improving clarity for prospective and first-year students while showcasing practical applied data science techniques.


### **Installing Dependencies:**
Before running any part of the RAG pipeline, we must install the required Python libraries.

These libraries support:

- Web scraping (requests, BeautifulSoup)

- Data handling (pandas, numpy)

- Vector embeddings (sentence-transformers)

- LLM inference (transformers, accelerate, bitsandbytes)

- Vector storage / retrieval (ChromaDB)

- Machine learning utilities (scikit-learn)

The following cell installs all dependencies needed to scrape data, build semantic chunks, store embeddings, and run the recommendation model.

In [1]:
!pip install \
    requests \
    beautifulsoup4 \
    pandas \
    lxml \
    numpy \
    scikit-learn \
    transformers \
    accelerate \
    chromadb \
    sentence-transformers


!pip install -U bitsandbytes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.1 MB/s et

### **Importing Necessary Packages:**
In this section, we import all required Python libraries for the full RAG pipeline.

In [2]:
# Web scraping
import requests
from bs4 import BeautifulSoup

# Data manipulation & utilities
import pandas as pd
import numpy as np
import re

# Machine learning utilities
from sklearn.metrics.pairwise import cosine_similarity

# Vector database
import chromadb
from chromadb.config import Settings

# Embedding model
from sentence_transformers import SentenceTransformer

# LLM (Gemma) + quantization
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import torch


## Scraping and Data storage

In this section, we scrape **official academic program pages** to collect information about:

- Majors  
- Program descriptions  
- Concentrations / focus areas  
- Career outcomes

### ISAT Program Description and Concentrations

In [3]:
# ISAT PROGRAM DESCRIPTION:
#Fetch the HTML content
ISAT_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27120#1').text

#Parse the HTML with BeautifulSoup
soup_isat = BeautifulSoup(ISAT_content, "lxml")

ISAT = soup_isat.find('div', class_="custom_leftpad_20")
desc_isat = ISAT.find('div', class_="acalog-core")
para_isat = desc_isat.find('p').get_text(separator=" ", strip=True)

# This shows the Program Description for ISAT
print(para_isat)

# Finds the Concentration description
inner_divs = ISAT.find_all('div', class_='custom_leftpad_20', recursive=False)
second_inner = inner_divs[1]
nested_divs = second_inner.find_all('div', class_='custom_leftpad_20', recursive=False)
second_nested = nested_divs[1]
acalog_cores = second_nested.find_all('div', class_='acalog-core', recursive=False)
target_div = acalog_cores[3]
texts = []
for tag in target_div.find_all(['p', 'ul']):
    texts.append(tag.get_text(separator=' ', strip=True))

Concentration_text = ' '.join(texts)

print(f"ISAT Concentrations: {Concentration_text}")

The Bachelor of Science degree in Integrated Science and Technology produces graduates broadly acquainted with basic science, technology and social science. All students pursue a common program through their sophomore year that provides a foundation of science and its technological applications. Studies are integrated to include mathematics, statistics, physics, chemistry, biology, computation, environmental science, modern production, energy, and the role of science and technology in society. During their junior and senior years, students pursue advanced studies in critical sectors of industry and government, including applied biotechnology, energy, environment and sustainability, industrial and manufacturing systems, public interest technology and science, and applied computing. Each student selects a concentration in these areas and pursues additional study that culminates in a capstone project. Students rely heavily upon systems thinking, analytical methods, and computation as prob

In [4]:
# ISAT APPLIED COMPUTING CONCENTRATION
#Fetch the HTML content
AppComp_content= requests.get('https://www.jmu.edu/cise/isat/academics/concentrations/applied-computing.shtml').text

#Parse the HTML with BeautifulSoup
soup_AppComp = BeautifulSoup(AppComp_content, "lxml")

AppComp = soup_AppComp.find('div', class_="tabular-row")
AppComp_desc = AppComp.find('div', class_="yui3-g-r")
# Get the outer 'yui3-u-1'
AppComp_outer = AppComp_desc.find('div', class_="yui3-u-1")

# Inside that outer one, find the two inner 'yui3-u-1' divs
AppComp_inners = AppComp_outer.find_all('div', class_="yui3-u-1")

# --- Extract description from the first inner div ---
AppComp_first = AppComp_inners[0]
AppComp_paras = AppComp_first.find_all('p')
AppComp_desc_text = ' '.join(p.get_text(separator=' ', strip=True) for p in AppComp_paras[1:5])

# --- Extract careers from the second inner div ---
AppComp_second = AppComp_inners[1]
AppComp_rwd = AppComp_second.find('div', class_='rwdwysiwyg')
AppComp_career_texts = [
    tag.get_text(separator=' ', strip=True)
    for tag in AppComp_rwd.find_all(['p', 'ul'])
]

# Handle bullet items better by joining list items with commas
AppComp_careers = []
for tag in AppComp_rwd.find_all(['p', 'ul']):
    if tag.name == 'ul':
        # Convert each <li> into a comma-separated phrase
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        AppComp_careers.append(', '.join(items))
    else:
        AppComp_careers.append(tag.get_text(separator=' ', strip=True))

AppComp_careers_text = ' '.join(AppComp_careers)

# --- Print both sections ---
print(f"\nISAT Applied Computing Description:\n{AppComp_desc_text}\n")
print(f"ISAT Applied Computing Careers:\n{AppComp_careers_text}")


ISAT Applied Computing Description:
You'll explore the vast realm of computing and its practical applications. From software development and data analysis to cybersecurity and artificial intelligence, this concentration covers a wide range of cutting-edge technologies that are shaping our world. Whether you dream of creating groundbreaking apps, developing intelligent systems, or protecting digital infrastructure, the Applied Computing concentration provides you with the skills and knowledge to excel. Technology is advancing at an unprecedented pace, and there has never been a more exciting time to be a part of it. In Applied Computing, you'll stay at the forefront of these advancements, acquiring the tools to tackle complex challenges and make a meaningful impact in various industries. We provide a comprehensive and hands-on approach. You'll have access to state-of-the-art facilities, expert faculty, and industry partnerships that enhance your learning experience. You'll have an oppo

In [5]:
# ISAT PUBLIC INTEREST CONCENTRATION
# Fetch the HTML content
PITS_content = requests.get('https://www.jmu.edu/cise/isat/academics/concentrations/public-interest-tech-science.shtml').text

# Parse the HTML with BeautifulSoup
soup_PITS = BeautifulSoup(PITS_content, "lxml")

PITS = soup_PITS.find('div', class_="tabular-row")
PITS_desc = PITS.find('div', class_="yui3-g-r")
# Get the outer 'yui3-u-1'
PITS_outer = PITS_desc.find('div', class_="yui3-u-1")

# Inside that outer one, find the two inner 'yui3-u-1' divs
PITS_inners = PITS_outer.find_all('div', class_="yui3-u-1")

# --- Extract description from the first inner div ---
PITS_first = PITS_inners[0]
PITS_paras = PITS_first.find_all('p')
PITS_desc_text = ' '.join(p.get_text(separator=' ', strip=True) for p in PITS_paras[1:5])

# --- Extract careers from the second inner div ---
PITS_second = PITS_inners[1]
PITS_rwd = PITS_second.find('div', class_='rwdwysiwyg')
PITS_career_texts = [
    tag.get_text(separator=' ', strip=True)
    for tag in PITS_rwd.find_all(['p', 'ul'])
]

# Handle bullet items better by joining list items with commas
PITS_careers = []
for tag in PITS_rwd.find_all(['p', 'ul']):
    if tag.name == 'ul':
        # Convert each <li> into a comma-separated phrase
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        PITS_careers.append(', '.join(items))
    else:
        PITS_careers.append(tag.get_text(separator=' ', strip=True))

PITS_careers_text = ' '.join(PITS_careers)

# --- Print both sections ---
print(f"\nISAT Public Interest Technology and Science Description:\n{PITS_desc_text}\n")
print(f"ISAT Public Interest Technology and Science Careers:\n{PITS_careers_text}")


ISAT Public Interest Technology and Science Description:
From environmental sustainability and healthcare accessibility to social justice, inclusion and equity – the PITS concentration equips you with the tools and knowledge to make a real difference in the world. You will Our faculty members are dedicated to mentoring and guiding you as you develop the skills, knowledge, and mindset necessary to become a catalyst for positive change. Together, we can address pressing social challenges, promote equality, and create a more just and sustainable world.

ISAT Public Interest Technology and Science Careers:
The demand for professionals with expertise in public interest technology and science is growing rapidly. Organizations across sectors, including nonprofits, government agencies, and social enterprises, are seeking individuals who can bridge the gap between technology, science, and societal needs. Potential careers include: Technology Policy Analyst, Data Privacy Advocate, Civic Technol

In [6]:
# ISAT INDUSTRIAL MANUFACTURING CONCENTRATION
# Fetch the HTML content
IMS_content = requests.get('https://www.jmu.edu/cise/isat/academics/concentrations/industrial-manufacturing-systems.shtml').text

# Parse the HTML with BeautifulSoup
soup_IMS = BeautifulSoup(IMS_content, "lxml")

IMS = soup_IMS.find('div', class_="tabular-row")
IMS_desc = IMS.find('div', class_="yui3-g-r")
# Get the outer 'yui3-u-1'
IMS_outer = IMS_desc.find('div', class_="yui3-u-1")

# Inside that outer one, find the two inner 'yui3-u-1' divs
IMS_inners = IMS_outer.find_all('div', class_="yui3-u-1")

# --- Extract description from the first inner div (include <p> and <ul>) ---
IMS_first = IMS_inners[0]

# Get both paragraphs and unordered lists
IMS_desc_tags = IMS_first.find_all(['p', 'ul'])

IMS_desc_parts = []
for tag in IMS_desc_tags:
    if tag.name == 'ul':
        # Flatten bullet list items into a comma-separated string
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        IMS_desc_parts.append(', '.join(items))
    else:
        # For paragraphs, just grab clean text
        IMS_desc_parts.append(tag.get_text(separator=' ', strip=True))

# Join them together into one description string
IMS_desc_text = ' '.join(IMS_desc_parts)

# --- Extract careers from the second inner div ---
IMS_second = IMS_inners[1]
IMS_rwd = IMS_second.find('div', class_='rwdwysiwyg')

# Get both paragraphs and unordered lists
IMS_career_texts = [
    tag.get_text(separator=' ', strip=True)
    for tag in IMS_rwd.find_all(['p', 'ul'])
]

# Handle bullet items better by joining list items with commas
IMS_careers = []
for tag in IMS_rwd.find_all(['p', 'ul']):
    if tag.name == 'ul':
        # Convert each <li> into a comma-separated phrase
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        IMS_careers.append(', '.join(items))
    else:
        IMS_careers.append(tag.get_text(separator=' ', strip=True))

IMS_careers_text = ' '.join(IMS_careers)

# --- Print both sections ---
print(f"\nISAT Industrial and Manufacturing Description:\n{IMS_desc_text}\n")
print(f"ISAT Industrial and Manufacturing Careers:\n{IMS_careers_text}")


ISAT Industrial and Manufacturing Description:
The Industrial & Manufacturing Systems (IMS) concentration , allows you to take a deep dive into the world of production and service systems. You’ll gain extensive knowledge and skills in Manufacturing, Automation, Operations management, Continuous improvement, Supply chain management You’ll be prepared to become a leader and collaborator in Industry 4.0. This is the era of sustainable, lean production systems where human and machine intelligence works together for mutual benefit. You will be equipped with the expertise to excel in industries ranging from artisanal craft production to complex global supply chains. Our graduates are innovators – capable of solving complex technical, economic, environmental, and social problems across a wide variety of sectors. From manufacturing and healthcare to logistics, information technology, and numerous service industries, graduates make a significant impact and drive positive change. If you're pass

In [7]:
# ISAT ENERGY CONCENTRATION
# Fetch the HTML content
Enr_content = requests.get('https://www.jmu.edu/cise/isat/academics/concentrations/energy.shtml').text

# Parse the HTML with BeautifulSoup
soup_Enr = BeautifulSoup(Enr_content, "lxml")

Enr = soup_Enr.find('div', class_="tabular-row")
Enr_desc = Enr.find('div', class_="yui3-g-r")

# Get the outer 'yui3-u-1'
Enr_outer = Enr_desc.find('div', class_="yui3-u-1")

# Inside that outer one, find the two inner 'yui3-u-1' divs
Enr_inners = Enr_outer.find_all('div', class_="yui3-u-1")

# --- Extract description from the first inner div (include <p> and <ul>) ---
Enr_first = Enr_inners[0]

# Get both paragraphs and unordered lists
Enr_desc_tags = Enr_first.find_all(['p', 'ul'])

Enr_desc_parts = []
for tag in Enr_desc_tags:
    if tag.name == 'ul':
        # Flatten bullet list items into a comma-separated string
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        Enr_desc_parts.append(', '.join(items))
    else:
        # For paragraphs, just grab clean text
        Enr_desc_parts.append(tag.get_text(separator=' ', strip=True))

# Join them together into one description string
Enr_desc_text = ' '.join(Enr_desc_parts)

# --- Extract careers from the second inner div ---
Enr_second = Enr_inners[1]
Enr_rwd = Enr_second.find('div', class_='rwdwysiwyg')

# Get both paragraphs and unordered lists
Enr_career_texts = [
    tag.get_text(separator=' ', strip=True)
    for tag in Enr_rwd.find_all(['p', 'ul'])
]

# Handle bullet items better by joining list items with commas
Enr_careers = []
for tag in Enr_rwd.find_all(['p', 'ul']):
    if tag.name == 'ul':
        # Convert each <li> into a comma-separated phrase
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        Enr_careers.append(', '.join(items))
    else:
        Enr_careers.append(tag.get_text(separator=' ', strip=True))

Enr_careers_text = ' '.join(Enr_careers)

# --- Print both sections ---
print(f"\nISAT Energy Concentration Description:\n{Enr_desc_text}\n")
print(f"ISAT Energy Concentration Careers:\n{Enr_careers_text}")


ISAT Energy Concentration Description:
In the Energy concentration, we empower students to contribute to a clean energy future. You will Learn how to apply the fundamentals of energy science to design renewable energy systems and enhance the efficiency of energy technologies while conserving resources. Explore how policy and economics influence energy demand, supply, and infrastructure. Understand the various commercial energy sources such as fossil, wind, solar, nuclear, geothermal, hydro, and tidal. You’ll also dive into the ethical considerations surrounding these energy sources. Gain hands-on experience though laboratory courses in data collection, analysis, instrumentation, teamwork, and effective communication of experimental results. Receive expert guidance from faculty who specialize in wind, solar, and geothermal energy sources. Be equipped with the knowledge and skills to become a leader in the energy industry. Develop a strong foundation in the sciences and technology relat

In [8]:
# ISAT APPLIED BIOTECHNOLOGY CONCENTRATION
# Fetch the HTML content
App_BT_content = requests.get('https://www.jmu.edu/cise/isat/academics/concentrations/applied-biotechnology.shtml').text

# Parse the HTML with BeautifulSoup
soup_App_BT = BeautifulSoup(App_BT_content, "lxml")

App_BT = soup_App_BT.find('div', class_="tabular-row")
App_BT_desc = App_BT.find('div', class_="yui3-g-r")

# Get the outer 'yui3-u-1'
App_BT_outer = App_BT_desc.find('div', class_="yui3-u-1")

# Inside that outer one, find the two inner 'yui3-u-1' divs
App_BT_inners = App_BT_outer.find_all('div', class_="yui3-u-1")

# --- Extract description from the first inner div (include <p> and <ul>) ---
App_BT_first = App_BT_inners[0]

# Get both paragraphs and unordered lists
App_BT_desc_tags = App_BT_first.find_all(['p', 'ul'])

App_BT_desc_parts = []
for tag in App_BT_desc_tags:
    if tag.name == 'ul':
        # Flatten bullet list items into a comma-separated string
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        App_BT_desc_parts.append(', '.join(items))
    else:
        # For paragraphs, just grab clean text
        App_BT_desc_parts.append(tag.get_text(separator=' ', strip=True))

# Join them together into one description string
App_BT_desc_text = ' '.join(App_BT_desc_parts)

# --- Extract careers from the second inner div ---
App_BT_second = App_BT_inners[1]
App_BT_rwd = App_BT_second.find('div', class_='rwdwysiwyg')

# Get both paragraphs and unordered lists
App_BT_career_texts = [
    tag.get_text(separator=' ', strip=True)
    for tag in App_BT_rwd.find_all(['p', 'ul'])
]

# Handle bullet items better by joining list items with commas
App_BT_careers = []
for tag in App_BT_rwd.find_all(['p', 'ul']):
    if tag.name == 'ul':
        # Convert each <li> into a comma-separated phrase
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        App_BT_careers.append(', '.join(items))
    else:
        App_BT_careers.append(tag.get_text(separator=' ', strip=True))

App_BT_careers_text = ' '.join(App_BT_careers)

# --- Print both sections ---
print(f"\nISAT Applied Biotechnology Concentration Description:\n{App_BT_desc_text}\n")
print(f"ISAT Applied Biotechnology Concentration Careers:\n{App_BT_careers_text}")



ISAT Applied Biotechnology Concentration Description:
The Applied Biotechnology concentration might be the perfect fit for you if You're fascinated by the intersection of biology and cutting-edge technology., You have a passion for exploring the potential of biotechnology to transform healthcare, agriculture, and the environment. You’ll delve into biotechnology and its practical applications – combining principles from biology, genetics, chemistry, and bioengineering to tackle real-world challenges and make groundbreaking discoveries. From developing new therapies and diagnostic tools to genetically modifying crops for sustainability – you'll be at the forefront of innovation. But what makes applied biotechnology truly exciting? The potential to improve human health, enhance food production, and address environmental issues. Imagine being part of a team that creates life-saving medications, engineers biofuels, or pioneers new methods for disease detection – making a meaningful impact 

In [9]:
# ISAT ENVIRONMENTAL CONCENTRATION
# Fetch the HTML content
Envr_content = requests.get('https://www.jmu.edu/cise/isat/academics/concentrations/environment.shtml').text

# Parse the HTML with BeautifulSoup
soup_Envr = BeautifulSoup(Envr_content, "lxml")

Envr = soup_Envr.find('div', class_="tabular-row")
Envr_desc = Envr.find('div', class_="yui3-g-r")

# Get the outer 'yui3-u-1'
Envr_outer = Envr_desc.find('div', class_="yui3-u-1")

# Inside that outer one, find the two inner 'yui3-u-1' divs
Envr_inners = Envr_outer.find_all('div', class_="yui3-u-1")

# --- Extract description from the first inner div (include <p> and <ul>) ---
Envr_first = Envr_inners[0]

# Get both paragraphs and unordered lists
Envr_desc_tags = Envr_first.find_all(['p', 'ul'])

Envr_desc_parts = []
for tag in Envr_desc_tags:
    if tag.name == 'ul':
        # Flatten bullet list items into a comma-separated string
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        Envr_desc_parts.append(', '.join(items))
    else:
        # For paragraphs, just grab clean text
        Envr_desc_parts.append(tag.get_text(separator=' ', strip=True))

# Join them together into one description string
Envr_desc_text = ' '.join(Envr_desc_parts)

# --- Extract careers from the second inner div ---
Envr_second = Envr_inners[1]
Envr_rwd = Envr_second.find('div', class_='rwdwysiwyg')

# Get both paragraphs and unordered lists
Envr_career_texts = [
    tag.get_text(separator=' ', strip=True)
    for tag in Envr_rwd.find_all(['p', 'ul'])
]

# Handle bullet items better by joining list items with commas
Envr_careers = []
for tag in Envr_rwd.find_all(['p', 'ul']):
    if tag.name == 'ul':
        # Convert each <li> into a comma-separated phrase
        items = [li.get_text(strip=True) for li in tag.find_all('li')]
        Envr_careers.append(', '.join(items))
    else:
        Envr_careers.append(tag.get_text(separator=' ', strip=True))

Envr_careers_text = ' '.join(Envr_careers)

# --- Print both sections ---
print(f"\nISAT Environment Concentration Description:\n{Envr_desc_text}\n")
print(f"ISAT Environment Concentration Careers:\n{Envr_careers_text}")


ISAT Environment Concentration Description:
Applying science and technology to understand environmental systems and solve today’s problems for a better tomorrow. Our planet faces various challenges, including climate change, fossil fuel consumption, resource depletion, pollution, industry impacts, transportation issues These factors affect the health and sustainability of both human and ecological systems. In the environment and sustainability concentration, we delve into these complex interactions and seek innovative solutions. Through a combination of hands-on fieldwork and lab-based courses, we apply a holistic problem-solving approach. Our curriculum equips you with the tools and knowledge to tackle today's environmental challenges head-on. We emphasize practical applications and real-world solutions to create a better tomorrow. Together, we can make a positive impact by conserving natural resources, improving industrial and agricultural practices, and fostering sustainability in 

### Computer Science Program Descriptions and Concentrations

In [10]:
# COMPUTER SCIENCE PROGRAM DESCRIPTION:
#Fetch the HTML content
CS_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27091#1').text

#Parse the HTML with BeautifulSoup
soup_cs = BeautifulSoup(CS_content, "lxml")

CS = soup_cs.find('div', class_="custom_leftpad_20")
desc_cs = CS.find('div', class_="acalog-core")
para_cs= desc_cs.find('p').get_text(separator=" ", strip=True)

print(para_cs)

#Focus Areas within CS
FA_CS_content = requests.get('https://www.jmu.edu/cise/cs/index.shtml').text
soup_FA_CS = BeautifulSoup(FA_CS_content, "lxml")

CS_Conc = []
area_blocks = soup_FA_CS.find_all("div", class_="rwdwysiwyg")

for block in area_blocks:
    # find titles (strong tags usually hold area names)
    strong = block.find("strong")
    name = strong.get_text(strip=True) if strong else "General Focus Area"

    # get rest of text
    desc = block.get_text(separator=" ", strip=True)

    CS_Conc.append({
        "name": name,
        "description": desc
    })

print("CS_Conc =", CS_Conc)



The Bachelor of Science degree in Computer Science prepares graduates to excel in the field of computing. All students complete required coursework in programming, mathematics, data structures, algorithms, software engineering, computer systems, and programming languages. Students also choose elective courses from a variety of computing subfields including robotics, artificial intelligence, human-computer interaction, cyber defense, database systems, and web applications. Throughout the curriculum, students apply their knowledge by completing many software development projects both individually and on a team, using a variety of languages and systems.
CS_Conc = [{'name': 'General Focus Area', 'description': 'The JMU Computer Science program is designed to nurture your passion for technology and equip you with the skills to thrive in this ever-evolving field. With small class sizes taught by our full-time faculty, state-of-the-art labs, and a curriculum at the forefront of industry trend

### IT Program Descriptions

In [11]:
# INFORMATION TECHNOLOGY PROGRAM DESCRIPTION:
#Fetch the HTML content
IT_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27154#1').text

#Parse the HTML with BeautifulSoup
soup_it = BeautifulSoup(IT_content, "lxml")

IT = soup_it.find('div', class_="custom_leftpad_20")
desc_it = IT.find('div', class_="acalog-core")
para_it= desc_it.find('p').get_text(separator=" ", strip=True)

print(para_it)



The Bachelor of Science degree in Information Technology focuses on highly relevant skills in cybersecurity, computer networking, and end-user design and development. The information technology degree goes beyond the science behind computers, teaching students how to design, develop, test and maintain solutions in a wide range of computing and networking application areas. Information technology studies are integrated to provide students with the knowledge and skills to meet the computer technology needs of business, government, healthcare, education and other organizations. In addition to core competencies in programming and networking, security and privacy, and other application-focused computing fields, the information technology major features a junior-level project to address a community need. Also, students in the information technology major are required to complete a two-semester senior capstone project, allowing them to apply the range of their abilities in a real-world contex

### Engineering Program Descriptions and concentrations

In [12]:
# ENGINERRING PROGRAM DESCRIPTION:
#Fetch the HTML content
ENGR_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27093#1').text

#Parse the HTML with BeautifulSoup
soup_engr= BeautifulSoup(ENGR_content, "lxml")

ENGR = soup_engr.find('div', class_="custom_leftpad_20")
desc_engr = ENGR.find('div', class_="acalog-core")
para_engr= desc_engr.find('p').get_text(separator=" ", strip=True)

print(para_engr)


#Concentrations within Engineering
Conc_Engr_content = requests.get('https://www.jmu.edu/cise/engineering/academics/concentrations.shtml').text

soup_Conc_Engr = BeautifulSoup(Conc_Engr_content, "lxml")

Engr_Conc= []

# Step 1: Target the second tabular-cell (correct container)
row = soup_Conc_Engr.find("div", class_="tabular-row")
cells = row.find_all("div", class_="tabular-cell")
second_cell = cells[1]

# Step 2: Find the internal wrapper
yui_outer = second_cell.find("div", class_="yui3-g-r")
yui_inner = yui_outer.find("div", class_="yui3-u-1")

# Step 3: Each concentration is its own block starting with <h3> or <h4>
sections = yui_inner.find_all(["h3", "h4"])

for sec in sections:
    title = sec.get_text(strip=True)  # concentration name
    content_parts = []

    # Get all siblings until next heading
    for sib in sec.find_next_siblings():
        if sib.name in ["h3", "h4"]:
            break
        if sib.name == "p":
            content_parts.append(sib.get_text(strip=True))
        elif sib.name == "ul":
            items = [li.get_text(strip=True) for li in sib.find_all("li")]
            content_parts.append(", ".join(items))

    full_desc = " ".join(content_parts)

    Engr_Conc.append({
        "name": title,
        "description": full_desc
    })

# PRINT RESULT
for c in Engr_Conc:
    print("\n", c["name"], "\n", c["description"])


JMU Engineering is an ABET accredited, 4-year interdisciplinary Bachelor of Science degree program. It offers a world-class undergraduate engineering experience based on a philosophy of learning by doing. It integrates many traditional engineering disciplines with course work in project management, engineering, design and liberal arts with foci on sustainable design and project delivery.

 Civil and Environmental 
 Are you passionate about shaping our world while protecting our environment? Our Civil and Environmental concentration prepares students with the skills to Analyze structural systems and design, Model and build infrastructure that abides by codes, specifications, and responsible stewardship of natural resources In this concentration, you'll delve into computational techniques for analyzing and modeling structures, master building codes and construction standards, and explore innovative sustainable practices. From designing steel and concrete infrastructure to implementing ec

### Intelligence Analysis Program Description

In [13]:
# IA PROGRAM DESCRIPTION:
#Fetch the HTML content
IA_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27215#1').text

#Parse the HTML with BeautifulSoup
soup_IA= BeautifulSoup(IA_content, "lxml")

IA = soup_IA.find('div', class_="custom_leftpad_20")
desc_IA = IA.find('div', class_="acalog-core")
paras_IA = desc_IA.find_all('p')
para_IA= " ".join(p.get_text(separator=" ", strip=True) for p in paras_IA)

print(para_IA)

The Bachelor of Science degree in Intelligence Analysis provides a multi-disciplinary education for students who seek careers as analysts, with a specialization in intelligence analysis. The degree integrates knowledge from a variety of academic disciplines (philosophy, history, political science, technology, business) and combines that with professionally oriented knowledge and skills. Students learn innovative ways to structure their thinking to assess complex real-world problems, along with how technology can be employed to acquire data, evaluate that data, and communicate it effectively to others.


### Geography Program Description and Concentration

In [14]:
# GEOGRAPHY PROGRAM DESCRIPTION:
#Fetch the HTML content
Geo_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27200#AppliedGeographicInformationScienceAGISConcentration').text

#Parse the HTML with BeautifulSoup
soup_Geo= BeautifulSoup(Geo_content, "lxml")

Geo = soup_Geo.find('div', class_="custom_leftpad_20")
desc_Geo = Geo.find('div', class_="acalog-core")
paras = desc_Geo.find_all('p')
para_Geo = " ".join(p.get_text(separator=" ", strip=True) for p in paras)
print(para_Geo)

#Concentrations within Geography
# --- Fetch and parse Geography Concentrations ---
Conc_Geo_content = requests.get('https://www.jmu.edu/cise/geography/academics/concentrations.shtml').text
soup_Conc_Geo = BeautifulSoup(Conc_Geo_content, "lxml")

Geo_Conc = []

blocks = soup_Conc_Geo.find_all("div", class_="rwdwysiwyg")

for block in blocks:
    title_tag = block.find_previous("h5")
    title = title_tag.get_text(strip=True) if title_tag else "General Concentration"

    desc = block.get_text(separator=" ", strip=True)

    Geo_Conc.append({
        "name": title,
        "description": desc
    })

print("Geo_Conc =", Geo_Conc)

Students complete a common core that gives a firm foundation in essential geographical knowledge which includes human geography, physical geography, geospatial techniques, statistics, and human/land relations. Beyond the core, concentrations encourage a deeper understanding of the discipline and the relevant skills to address critical problems faced by humanity. These concentrations can be customized to the student’s interests (Custom) or focus on a particular area within Geography (Applied Geographic Information Science; Environment, Conservation, Sustainability and Development).
Geo_Conc = [{'name': 'Applied Geographic Information Science\xa0Concentration', 'description': 'In the Applied Geographic Information Science concentration, you’ll focus on the practical application of geographic information systems (GIS) and spatial analysis techniques to solve real-world problems and address spatially-related challenges. You’ll combine GIS, computer technology, and spatial analysis methods 

### Biotechnology Program Description and Concentration

In [15]:
# BIOTECHNOLOGY PROGRAM DESCRIPTION:
#Fetch the HTML content
BioT_content= requests.get('https://catalog.jmu.edu/preview_program.php?catoid=62&poid=27081#1').text

#Parse the HTML with BeautifulSoup
soup_BioT= BeautifulSoup(BioT_content, "lxml")

BioT = soup_BioT.find('div', class_="custom_leftpad_20")
desc_BioT = BioT.find('div', class_="acalog-core")

paras_BioT = desc_BioT.find_all('p')
selected_BioT = paras_BioT[1:5]
para_BioT = " ".join(p.get_text(separator=" ", strip=True) for p in selected_BioT)

def clean_text(text):
    # Replace non-breaking spaces with regular spaces
    text = text.replace("\xa0", " ")
    # Optionally collapse multiple spaces into one
    text = " ".join(text.split())
    return text

# Example
para_BioT_clean = clean_text(para_BioT)

print(para_BioT_clean)



JMU Biotechnology is a 4-year interdisciplinary major leading to a bachelor of science degree. The program is shared among the Departments of Biology , Chemistry and Biochemistry and Integrated Science and Technology (ISAT) . Students undertake a rigorous curriculum rich with hands-on laboratory experiences, critical analyses of both the “how” and the “why” of biotechnological solutions, and the development of transferable skills needed to keep up in a rapidly changing field. The combination of both scientific/technical depth and cross-disciplinary breadth allow graduates to pursue diverse career paths in industry, government and advanced studies. Biotechnology majors complete 47-50 credit hours of science foundation courses,17 credit hours of biotechnology transition and core courses, and 15 credit hours of electives. Advanced courses are selected from an extensive list of Biology , Chemistry and Biochemistry , ISAT , Engineering and Mathematics and Statistics offerings. Students work

### **Storing the Scraped Data**
This section organizes the scraped program information into a structured Python list.  
Each entry contains the program name, its main description, and any concentrations or focus areas.

The resulting `Scraped_data` object is then saved as a JSON file for downstream processing.

In [16]:
Scraped_data = [
    {
        "program": "ISAT",
        "description": para_isat,
        "concentrations": [
            {"name": "Applied Computing", "description": AppComp_desc_text, "careers": AppComp_careers_text},
            {"name": "Public Interest Technology and Science", "description": PITS_desc_text, "careers": PITS_careers_text},
            {"name": "Industrial Manufacturing Systems", "description": IMS_desc_text, "careers": IMS_careers_text},
            {"name": "Energy", "description": Enr_desc_text, "careers": Enr_careers_text},
            {"name": "Applied Biotechnology", "description": App_BT_desc_text, "careers": App_BT_careers_text},
            {"name": "Applied Biotechnology", "description": Envr_desc_text,"careers": Envr_careers_text }
        ]
    },
    {
        "program": "Computer Science",
        "description": para_cs,
        "focus_areas": CS_Conc
    },
    {
        "program": "Information Technology",
        "description": para_it,
    },
    {
        "program": "Engineering",
        "description": para_engr,
        "concentrations": Engr_Conc
    },
    {
        "program": "Intelligence Analysis",
        "description": para_IA
    },
    {
        "program": "Geography",
        "description": para_Geo,
        "concentrations": Geo_Conc
    },
    {
        "program": "Biotechnology",
        "description": para_BioT_clean   }
]


In [17]:
import json

with open("CISE_programs.json", "w", encoding="utf-8") as f:
    json.dump(Scraped_data, f, ensure_ascii=False, indent=2)



## **Cleaning Program Text:**
This section standardizes and cleans all scraped descriptions by removing extra spaces and encoded characters.
The goal is to prepare clean, consistent text for chunking and embedding.

In [19]:
def clean_text(text):
    text = text.replace("\xa0", " ")
    text = " ".join(text.split())
    return text

for item in Scraped_data:
    item["description"] = clean_text(item.get("description", ""))
    if "concentrations" in item:
        for conc in item["concentrations"]:
            conc["description"] = clean_text(conc.get("description", ""))
            conc["careers"] = clean_text(conc.get("careers", ""))
    if "focus_areas" in item:
        for fa in item["focus_areas"]:
            if isinstance(fa, dict):
                fa["description"] = clean_text(fa.get("description", ""))


## **Semantic Chunking and Embedding:**
This section breaks long descriptions into smaller, meaningful text chunks
so the retrieval system can match user questions more accurately.

In [20]:
import re

def semantic_split(text, max_chunk_size=350):
    """
    Very simple semantic chunking:
      - splits at sentences
      - groups sentences together until a chunk reaches ~350 chars
    """
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) <= max_chunk_size:
            current += " " + sent
        else:
            chunks.append(current.strip())
            current = sent

    if current.strip():
        chunks.append(current.strip())

    return chunks


def create_chunks(Scraped_data):
    chunks = []

    for item in Scraped_data:
        program_name = item.get("program", "").strip()

        # ----- PROGRAM OVERVIEW -----
        overview = item.get("description", "").strip()
        if overview:
            for i, subchunk in enumerate(semantic_split(overview)):
                chunks.append({
                    "id": f"{program_name}_overview_{i}",
                    "program": program_name,
                    "type": "overview",
                    "text": subchunk
                })

        # ----- CONCENTRATIONS -----
        if "concentrations" in item:
            for conc in item["concentrations"]:
                c_name = conc.get("name", "")
                c_text = (conc.get("description", "") + " " + conc.get("careers","")).strip()
                for i, subchunk in enumerate(semantic_split(c_text)):
                    chunks.append({
                        "id": f"{program_name}_{c_name}_{i}".replace(" ", "_"),
                        "program": program_name,
                        "type": "concentration",
                        "text": subchunk
                    })

        # ----- FOCUS AREAS -----
        if "focus_areas" in item:
            for fa in item["focus_areas"]:
                for i, subchunk in enumerate(semantic_split(fa["description"])):
                    chunks.append({
                        "id": f"{program_name}_{fa['name']}_{i}".replace(" ", "_"),
                        "program": program_name,
                        "type": "focus_area",
                        "text": subchunk
                    })

    return chunks


# RUN IT
chunks = create_chunks(Scraped_data)
print("Generated", len(chunks), "semantic chunks.")


Generated 83 semantic chunks.


This section below converts each chunk into an embedding and saves it inside a persistent Chroma vector database for fast semantic retrieval.

In [21]:

# -------------------------
# Store chunks in Chroma
# -------------------------
from sentence_transformers import SentenceTransformer
import chromadb

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Create persistent Chroma client
chroma = chromadb.PersistentClient(path="./vector_store")

# Create/get collection
collection = chroma.get_or_create_collection(
    name="program_chunks",
    metadata={"hnsw:space": "cosine"}
)

# Add chunks to Chroma
for chunk in chunks:
    text = chunk["text"]
    emb = embedding_model.encode(text, convert_to_numpy=True).tolist()

    collection.add(
        ids=[chunk["id"]],
        embeddings=[emb],
        documents=[text],
        metadatas=[{
            "program": chunk["program"],
            "type": chunk["type"],
            "snippet": text[:200]
        }]
    )

print("Done storing to Chroma!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Done storing to Chroma!


### Hugging Face Authentication
Before loading the Gemma model, the notebook authenticates with Hugging Face so it can access gated or protected model files. This command opens a login prompt where you enter your Hugging Face access token.

In [22]:
!hf auth login #---- Take out if running through Jupyter NB


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `Gemma3RAGPrototype` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `Gemma3

### **RAG Pipeline:**
This section brings together the full Retrieval-Augmented Generation workflow.
First, the Gemma-3-1B model, tokenizer, and embedding model are loaded, along with the Chroma vector store.

Next, the user’s question is embedded and used to retrieve both high-level program overviews and relevant concentration descriptions.

All retrieved text is then combined into a structured context block, injected into a controlled system prompt, and passed to the LLM for final answer generation.

This single cell performs the complete pipeline: setup → retrieval → prompt construction → generation → response.

This section runs the complete inference pipeline using:

- **Gemma-3-1B-IT** (quantized in 8-bit)
- **MiniLM sentence transformer** for embeddings
- **ChromaDB** for vector retrieval  

The model runs fully in Google Colab with GPU acceleration.  
Below this cell, a **Local GPU model** and **HF-token local loading option** are provided (commented out) so that it may run or inspect the system locally if desired.


In [23]:
# ------------------------------
# Accessing Model
# ------------------------------
model_id = "google/gemma-3-1b-it"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma = chromadb.PersistentClient(path="./vector_store")
collection = chroma.get_or_create_collection(
    name="program_chunks",
    metadata={"hnsw:space": "cosine"}
)

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)

# -------------------------------------------------
#  Local GPU run:
# -------------------------------------------------
# If running locally, must:
# 1. Have a CUDA-enabled GPU
# 2. Have access to the gated Gemma repo
# 3. Set environment variable:  export HF_TOKEN=xxxx
#

"""
# --- Uncomment ONLY when running locally with GPU ---

import os
HF_TOKEN = os.getenv("HF_TOKEN")   #  load token securely

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,                # authenticates access
    quantization_config=quantization_config,
    device_map="auto"              # auto-detects GPU
).eval()

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN
)
"""

# ------------------------------
# User Question
# ------------------------------
user_question = (
    "I am a prospective student who is interested in anything related to computers, "
    "but I don’t want to do Computer Science. Is there any options for me? "
    "I am looking for a major with multiple concentrations, particularly to focus on science, technology and society."

)


# ------------------------------
# Retrieval
# ------------------------------
question_embedding = embedding_model.encode(user_question).tolist()

# ------------------------------
# Retrieve overviews
# ------------------------------
overview_results = collection.get(
    where={"type": "overview"},
    include=["documents", "metadatas"]
)

overview_docs = overview_results["documents"]
overview_meta = overview_results["metadatas"]

# ------------------------------
# Retrieve concentration matches using embeddings
# ------------------------------
question_embedding = embedding_model.encode(user_question).tolist()

concentration_results = collection.query(
    query_embeddings=[question_embedding],
    n_results=5,
    where={"type": "concentration"},
    include=["documents", "metadatas"]
)

concentration_docs = concentration_results["documents"][0]
concentration_meta = concentration_results["metadatas"][0]

# ------------------------------
# Combine context
# ------------------------------
retrieved_context = "\n".join(overview_docs + concentration_docs)


#print("\n--- RETRIEVED CONTEXT  ---")
#print(retrieved_context) #to check what is being retrieved
#print("---------------------------------------")


system_instruction = (
    "Answer the student's question using ONLY the information provided below.\n\n"
    "You ARE allowed to:\n"
      "- Compare the majors in the provided text.\n"
      "- Decide which major is the best match based on the descriptions.\n"
      "- Mention concentrations ONLY as supporting details.\n"
      "- Use reasonable interpretation of the descriptions.\n\n"
    "You are NOT allowed to:\n"
      "- Invent new majors.\n"
      "- Rename majors.\n"
      "- Treat concentrations as majors.\n"
      "- Use information not present in the provided text.\n\n"
    "IMPORTANT:\n"
    "If a program name is NOT explicitly written in the provided text, you MUST NOT include it in your answer.\n"
    "If multiple majors are listed, choose the ONE that best fits the student’s interests.\n"

)



final_prompt = f"""
{system_instruction}

--- CONTEXT ---
{retrieved_context}
---

STUDENT QUESTION: {user_question}

Answer:
"""


# ------------------------------
# Generation
# ------------------------------
messages = [{"role": "user", "content": final_prompt}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300
    )

decoded_outputs = tokenizer.decode(outputs[0], skip_special_tokens=True)

response_text = decoded_outputs.replace(final_prompt, "").strip()
response_text = (
    response_text.replace("user\n", " ").replace("model\n", "").strip()
)

print("\n--- RAG MODEL RESPONSE ---")
print(response_text)


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]


--- RAG MODEL RESPONSE ---
userBased on the provided text, you could consider the following options:

1.  **Information Technology:** This major integrates knowledge from a variety of academic disciplines (philosophy, history, political science, technology, business) and combines that with professionally oriented knowledge and skills. It offers a strong foundation in cybersecurity, computer networking, and end-user design and development.

2.  **Intelligence Analysis:** This major focuses on analyzing complex real-world problems and leveraging technology to acquire data, evaluate that data, and communicate it effectively.

3.  **Applied Geographic Information Science:** This concentration focuses on the application of geographic information systems to address critical problems faced by humanity, with a focus on sustainable design and project delivery.

4.  **Biotechnology:** This major integrates knowledge from a variety of academic disciplines (biology, chemistry, biochemistry) and c